In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Kafka Example").getOrCreate()

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.endpoint", "s3.ap-northeast-2.amazonaws.com")
hconf.set("fs.s3a.path.style.access", "false")
hconf.set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")

26/01/10 14:32:48 WARN Utils: Your hostname, genius.local resolves to a loopback address: 127.0.0.1; using 10.12.2.14 instead (on interface en0)
26/01/10 14:32:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/10 14:32:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
dataframe = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka-broker-1:9093,kafka-broker-2:9096,kafka-broker-3:9099") \
    .option("kafka.security.protocol", "SASL_PLAINTEXT") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", "org.apache.kafka.common.security.plain.PlainLoginModule ""required username=admin password=admin;") \
    .option("subscribe", "mmix-product-topic") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

In [4]:
# dataframe = spark \
#     .readStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", "kafka-broker-1:9093,kafka-broker-2:9096,kafka-broker-3:9098") \
#     .option("subscribe", "mmix-product-topic") \
#     .option("startingOffsets", "earliest") \
#     .option("failOnDataLoss", "false") \
#     .load()

In [5]:
# q = dataframe \
#     .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value") \
#     .writeStream.format("console").option("truncate", "false") \
#     .trigger(processingTime="5 seconds") \
#     .start()
# q.awaitTermination()

In [ ]:
q = dataframe \
    .selectExpr("topic", "CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "timestamp") \
    .writeStream \
    .format("parquet") \
    .option("path", "s3a://mmix-prod-dataengineer-datalakehouse/streaming/product/") \
    .option("checkpointLocation", "s3a://mmix-prod-dataengineer-datalakehouse/streaming/checkpoints/product/") \
    .option("compression", "snappy") \
    .trigger(processingTime="10 seconds") \
    .start()

q.awaitTermination()

26/01/10 14:32:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/01/10 14:32:50 WARN VersionInfoUtils: The AWS SDK for Java 1.x reached end of support on December 31, 2025. For more information, see https://aws.amazon.com/blogs/developer/the-aws-sdk-for-java-1-x-is-in-maintenance-mode-effective-july-31-2024/
You can print where on the file system the AWS SDK for Java 1.x core runtime is located by setting the AWS_JAVA_V1_PRINT_LOCATION environment variable or aws.java.v1.printLocation system property to 'true'.
This message can be disabled by setting the AWS_JAVA_V1_DISABLE_DEPRECATION_ANNOUNCEMENT environment variable or aws.java.v1.disableDeprecationAnnouncement system property to 'true'.
The AWS SDK for Java 1.x is being used here:
at java.base/java.lang.Thread.getStackTrace(Thread.java:2451)
at com.amazonaws.util.VersionInfoUtils.printDeprecationAnnouncement(VersionInfoUtils.java:81)
at com.amazonaws.u